In [33]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from torch.nn.utils.rnn import pad_sequence
import re
import string
from collections import Counter
import gc
import numpy as np
from opacus import PrivacyEngine
from opacus.layers import DPLSTM
from privacy_analysis.compute_privacy_sgm import compute_dp_sgd_privacy

In [34]:
imdb = load_dataset("imdb")
cornellmovie = load_dataset("cornell-movie-review-data/rotten_tomatoes")

max_imdb_contribution = len(imdb["train"])
max_cornellmovie_contribution = len(cornellmovie["train"])

total_if_imdb_maxed = int(len(imdb["train"]) / 0.96)
total_if_cornellmovie_maxed = int(len(cornellmovie["train"]) / 0.04)
total_desired_size = min(total_if_imdb_maxed, total_if_cornellmovie_maxed)
imdb_size = int(0.96 * total_desired_size)
cornellmovie_size = int(0.04 * total_desired_size)

np.random.seed(42)
imdb_indices = np.random.choice(len(imdb["train"]), size=len(imdb), replace=False)
cornellmovie_indices = np.random.choice(len(cornellmovie["train"]), size=6250, replace=False)

imdb_subset = imdb["train"]
cornellmovie_subset = cornellmovie["train"].select(cornellmovie_indices)

In [35]:
def clean_text(text):
    text = text.lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text):
    return clean_text(text).split()

counter = Counter()
for example in imdb["train"]:
    counter.update(tokenize(example["text"]))

min_freq = 3
vocab = {"<pad>": 0, "<unk>": 1}
for word, freq in counter.items():
    if freq >= min_freq:
        vocab[word] = len(vocab)
inv_vocab = {v: k for k, v in vocab.items()}
PAD_IDX = vocab["<pad>"]
UNK_IDX = vocab["<unk>"]

def text_pipeline(text, max_len=256):
    tokens = tokenize(text)[:max_len]
    bow_vector = torch.zeros(len(vocab), dtype=torch.float)
    for token in tokens:
        idx = vocab.get(token, UNK_IDX)
        bow_vector[idx] += 1
    return bow_vector

In [36]:
class TextDataset(Dataset):
    def __init__(self, hf_dataset, label_transform, text_field, max_len=256):
        self.data = hf_dataset
        self.label_transform = label_transform
        self.max_len = max_len
        self.text_field = text_field

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        tokens = text_pipeline(item[self.text_field], max_len=self.max_len)
        label = torch.tensor(self.label_transform(item["label"]), dtype=torch.long)
        return tokens, label

def collate_batch(batch):
    texts, labels = zip(*batch)
    padded = pad_sequence(texts, batch_first=True, padding_value=PAD_IDX)
    return padded, torch.stack(labels)

In [37]:
train_loader_public = DataLoader(
    TextDataset(cornellmovie_subset, label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=True, collate_fn=collate_batch
)

train_loader_private = DataLoader(
    TextDataset(imdb_subset, label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=True, collate_fn=collate_batch
)

test_loader = DataLoader(
    TextDataset(imdb["test"], label_transform=lambda x: x, text_field="text"),
    batch_size=64, shuffle=False, collate_fn=collate_batch
)

In [38]:
class Model(nn.Module):
    def __init__(self, vocab_size, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(vocab_size, num_classes)

    def forward(self, x):
        return self.fc(x)

In [39]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Model(len(vocab), num_classes=2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)
privacy_engine = PrivacyEngine()
gc.collect()

/opt/anaconda3/lib/python3.12/site-packages/opacus/privacy_engine.py:96: UserWarning: Secure RNG turned off. This is perfectly fine for experimentation as it allows for much faster training performance, but remember to turn it on and retrain one last time before production with ``secure_mode`` turned on.
  warnings.warn(


153

In [40]:
def PACED_SGD(model, D_loader, D_s_loader, test_loader, lr, sigma, l, T, delta):
    """
    Parameters:
        D_loader: private training data loader
        n: private batch size
        D_s_loader: public training data loader
        n_s: public batch size
        lr: learning rate
        sigma: noise scale
        l: loss function
        T: number of training iterations
    """
    
    train_losses = []
    train_accuracies = []
    test_losses = []
    test_accuracies = []

    n_private = len(D_loader.dataset)
    batch_size_private = D_loader.batch_size
    
    optim = torch.optim.SGD(model.parameters(), lr=lr)
    pub_iter = iter(D_s_loader)

    C = 0

    for epoch in range(T):
        model.train()
        total_loss = 0.0
        correct = 0
        total = 0

        for priv_inputs, priv_labels in D_loader:
            
            try:
                pub_inputs, pub_labels = next(pub_iter)
            except StopIteration:
                pub_iter = iter(D_s_loader)
                pub_inputs, pub_labels = next(pub_iter) 
            
            optim.zero_grad()
            pub_out = model(pub_inputs)
            pub_loss = l(pub_out, pub_labels)
            pub_loss.backward()
            
            pub_grad_norm = 0.0
            for p in model.parameters():
                if p.grad is not None:
                    pub_grad_norm += p.grad.norm().item() ** 2
            pub_grad_norm = pub_grad_norm ** 0.5

            pub_grad = []
            for p in model.parameters():
                pub_grad.append(p.grad.detach().clone())
            pub_grad = [g.clone() for g in pub_grad]
            
            grads = [torch.zeros_like(p) for p in model.parameters()]
            noisy_grad_norms = []
            
            for i, (x, y) in enumerate(zip(priv_inputs, priv_labels)):
                optim.zero_grad()
                output = model(x.unsqueeze(0))
                loss = l(output, y.unsqueeze(0))
                loss.backward()
                
                noisy_norm = 0
                for j, p in enumerate(model.parameters()):
                    if p.grad is not None:
                        grads[j] += p.grad.clone()
                        noisy_norm += p.grad.norm().item() * (pub_grad_norm)/(max(pub_grad_norm, p.grad.norm().item())) + torch.normal(0, pub_grad_norm * sigma, (1,)).item()          
                noisy_grad_norms.append(noisy_norm)
            
            avg_pub_grad_norm = pub_grad_norm
            avg_noisy_priv_grad_norm = sum(noisy_grad_norms) / len(noisy_grad_norms)
            
            C = 0.3 * avg_pub_grad_norm + 0.7 * avg_noisy_priv_grad_norm

            C = 1.3 * C
            
            total_grad_norm = 0.0
            for g in grads:
                total_grad_norm += g.norm().item() ** 2
            total_grad_norm = total_grad_norm ** 0.5
            
            clip_factor = min(1.0, C / (total_grad_norm + 1e-8))
            
            for i, g in enumerate(grads):
                grads[i] = g * clip_factor
                grads[i] += torch.normal(0, sigma * C, g.shape)
            
            optim.zero_grad()
            for p, g in zip(model.parameters(), grads):
                if p.requires_grad:
                    p.grad = g / len(priv_inputs)
            
            optim.step()

            total_loss += pub_loss.item() * priv_inputs.size(0)
            with torch.no_grad():
                priv_out = model(priv_inputs)
                _, predicted = torch.max(priv_out.data, 1)
                total += priv_labels.size(0)
                correct += (predicted == priv_labels).sum().item()
        
        epoch_loss = total_loss / total
        epoch_accuracy = 100 * correct / total

        model.eval()
        test_correct = 0
        test_total = 0
        test_loss = 0.0

        with torch.no_grad():
            for inputs, labels in test_loader:
                outputs = model(inputs)
                loss = l(outputs, labels)
                test_loss += loss.item() * labels.size(0)
                _, predicted = torch.max(outputs.data, 1)
                test_total += labels.size(0)
                test_correct += (predicted == labels).sum().item()

        test_accuracy = 100 * test_correct / test_total
        test_loss = test_loss / test_total

        train_losses.append(epoch_loss)
        train_accuracies.append(epoch_accuracy)
        test_losses.append(test_loss)
        test_accuracies.append(test_accuracy)

        eps, _ = compute_dp_sgd_privacy(n_private, batch_size_private, sigma, epoch + 1, delta)
        print(f"Epoch [{epoch+1}/{T}] ... Training Loss: {epoch_loss:.4f} ... Training Accuracy: {epoch_accuracy:.2f}% ... Test Loss: {test_loss:.4f} ... Test Accuracy: {test_accuracy:.2f}% ... Epsilon: {eps:.4f} ... Delta: {delta}")

    return train_losses, train_accuracies, test_losses, test_accuracies

In [41]:
train_losses, train_accuracies, test_losses, test_accuracies = PACED_SGD(
                                                                        model=model,
                                                                        D_loader=train_loader_private,
                                                                        D_s_loader=train_loader_public,
                                                                        test_loader=test_loader,
                                                                        lr=0.01, 
                                                                        sigma=0.716, 
                                                                        l=criterion,
                                                                        T=15,
                                                                        delta=1e-5
                                                                      )

DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.716 iterated over 391 steps satisfies differential privacy with eps = 1.95 and delta = 1e-05.
The optimal RDP order is 6.0.
Epoch [1/15] ... Training Loss: 0.6926 ... Training Accuracy: 55.62% ... Test Loss: 0.6839 ... Test Accuracy: 57.92% ... Epsilon: 1.9501 ... Delta: 1e-05
DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.716 iterated over 782 steps satisfies differential privacy with eps = 2.12 and delta = 1e-05.
The optimal RDP order is 5.5.
Epoch [2/15] ... Training Loss: 0.6912 ... Training Accuracy: 61.24% ... Test Loss: 0.6744 ... Test Accuracy: 62.72% ... Epsilon: 2.1208 ... Delta: 1e-05
DP-SGD with sampling rate = 0.256% and noise_multiplier = 0.716 iterated over 1172 steps satisfies differential privacy with eps = 2.19 and delta = 1e-05.
The optimal RDP order is 5.5.
Epoch [3/15] ... Training Loss: 0.6903 ... Training Accuracy: 63.15% ... Test Loss: 0.6685 ... Test Accuracy: 62.94% ... Epsilon: 2.1915 ..